# Ders 01 - Yapay Zeka Ajanlarına Giriş

**Yeni Başlayanlar için Yapay Zeka Ajanları** kursunun ilk dersine hoş geldiniz!

**Yapay zeka ajanı**, büyük bir dil modelini (LLM) akıl yürütme motoru olarak kullanan ve gerçek dünyada *eylemler* gerçekleştirebilen — API çağrıları yapma, veritabanları sorgulama veya kod çalıştırma gibi — kullanıcı adına bir hedefi başarmak için programdır.

Bu not defterinde ilk ajanınızı oluşturacaksınız: tatil destinasyonları öneren bir **Seyahat Ajanı**. Bu süreçte şunları öğreneceksiniz:

1. **Microsoft Agent Framework** kullanarak Azure AI Foundry Agent Servisi'ne bağlanmayı.
2. Ajana çağırabileceği bir **araç** — basit bir Python fonksiyonu vermeyi.
3. Ajani çalıştırmayı ve yanıtını incelemeyi.
4. Ajanın yanıtını token token akış olarak görüntülemeyi.


## Kurulum

Bu not defresini çalıştırmadan önce, şunlara sahip olduğunuzdan emin olun:

1. **Dağıtılmış bir sohbet modeli olan bir Azure AI Foundry projesi** (örneğin `gpt-4o-mini`).
2. **Azure CLI ile giriş yapmış olmak** — terminalinizde `az login` komutunu çalıştırın.
3. **Gerekli ortam değişkenlerini ayarlamış olmak:**
   - `AZURE_AI_PROJECT_ENDPOINT` — Azure AI Foundry proje uç noktanız.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — dağıtılmış modelinizin adı.

Aşağıdaki hücre ihtiyacınız olan Python paketlerini yükler.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## İlk Ajanınızı Yaratmak

Bir ajan iki şeye ihtiyaç duyar:

- Ona *kim olduğunu* ve *nasıl davranacağını* söyleyen **Talimatlar** (bir sistem istemi).
- Ajanın bilgi almak veya eylem gerçekleştirmek için çağırabileceği `@tool` ile süslenmiş Python fonksiyonları olan **Araçlar**.

Aşağıda popüler tatil yerleri listesini döndüren basit bir araç tanımlıyoruz. Kullanıcı seyahat önerileri istediğinde ajan bu aracı kullanacak.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Akış Yanıtları

Daha etkileşimli bir deneyim için ajan yanıtını **akış** şeklinde alabilirsiniz. Tam cevabı beklemek yerine, ajan oluşturdukça metin parçacıkları verir. Bu, çıktıyı gerçek zamanlı göstermek istediğiniz sohbet arayüzlerinde özellikle faydalıdır.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Özet

Bu derste şunları öğrendiniz:

- `AzureAIProjectAgentProvider` aracılığıyla Azure AI Foundry Agent Service'e bağlanan **bir sağlayıcı oluşturmayı**.
- Ajanın Python fonksiyonlarınızı çağırabilmesi için `@tool` dekoratörü kullanarak **bir araç tanımlamayı**.
- Kullanıcı mesajıyla **ajanı çalıştırmayı** ve yanıtını yazdırmayı.
- Gerçek zamanlı çıktı için **yanıtları akış halinde almayı**.

Sonraki derste, ajanik çerçeveleri daha derinlemesine inceleyecek ve ajanlara daha güçlü araçlar ile çok adımlı akıl yürütme yetenekleri vermeyi öğreneceğiz.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Feragatname**:
Bu belge, AI çeviri hizmeti [Co-op Translator](https://github.com/Azure/co-op-translator) kullanılarak çevrilmiştir. Doğruluk için çaba sarf etsek de, otomatik çevirilerin hata veya yanlışlık içerebileceğini lütfen unutmayınız. Orijinal belge, kendi dilinde yetkili kaynak olarak kabul edilmelidir. Kritik bilgiler için profesyonel insan çevirisi önerilir. Bu çevirinin kullanımı sonucu ortaya çıkabilecek yanlış anlamalardan veya yanlış yorumlamalardan sorumlu değiliz.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
